In [ ]:
# Секция 1. Облачное GPU-окружение: установка зависимостей, OpenMM, CUDA и nvidia-smi.
# В Colab/DataSphere запускайте эту ячейку первой.

import os
import sys
import subprocess
import shutil
import time
import math
import json
import warnings
from pathlib import Path
from statistics import mean, stdev
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "openmm", "numpy", "pandas", "tqdm"
])

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import openmm as mm
import openmm.app as app
from openmm import unit

print("Python:", sys.version.split()[0])
print("OpenMM:", mm.version.version)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "<not set>"))

print("\n===== nvidia-smi =====")
if shutil.which("nvidia-smi"):
    smi = subprocess.run(["nvidia-smi"], text=True, capture_output=True)
    print(smi.stdout if smi.stdout else smi.stderr)
else:
    print("nvidia-smi not found")

OPENMM_PLATFORMS = [mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]
CUDA_AVAILABLE = "CUDA" in OPENMM_PLATFORMS

print("\nOpenMM platforms:", OPENMM_PLATFORMS)
print("CUDA visible to OpenMM:", CUDA_AVAILABLE)

if not CUDA_AVAILABLE:
    warnings.warn(
        "CUDA не видна OpenMM. Ноутбук может работать на CPU, "
        "но CPU-результаты нельзя использовать для оценки времени DataSphere GPU-run.",
        RuntimeWarning,
    )

def query_gpu_info():
    if not shutil.which("nvidia-smi"):
        return []
    cmd = [
        "nvidia-smi",
        "--query-gpu=index,name,memory.total,driver_version",
        "--format=csv,noheader,nounits",
    ]
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.returncode != 0:
        return []
    rows = []
    for line in result.stdout.strip().splitlines():
        parts = [p.strip() for p in line.split(",")]
        if len(parts) >= 4:
            rows.append({
                "index": int(parts[0]),
                "name": parts[1],
                "memory_total_mb": parts[2],
                "driver_version": parts[3],
            })
    return rows

GPU_INFO = query_gpu_info()
GPU_NAMES = {row["index"]: row["name"] for row in GPU_INFO}
AVAILABLE_GPU_INDICES = [row["index"] for row in GPU_INFO]

print("\nGPU_INFO:")
if GPU_INFO:
    print(pd.DataFrame(GPU_INFO))
else:
    print("No GPUs reported by nvidia-smi")

if CUDA_AVAILABLE:
    try:
        cuda_platform = mm.Platform.getPlatformByName("CUDA")
        print("\nCUDA platform properties:", list(cuda_platform.getPropertyNames()))
    except Exception as exc:
        print("Could not inspect CUDA platform:", repr(exc))


# LJ/OpenMM GPU Benchmarks

Самодостаточный бенчмарковый ноутбук для подготовки будущего большого запуска EOS-точек в Yandex DataSphere. Он не строит финальные EOS-графики, не фитит уравнение Ван-дер-Ваальса, не сохраняет траектории, координаты, профили плотности, картинки или видео.

Результат ноутбука — CSV-таблицы в benchmark_data/, по которым выбираются N, equil_steps, prod_steps, стратегия multi-GPU и параметры будущего acquisition notebook.


In [ ]:
# BENCHMARK_PARAMS
# Все параметры бенчмарков задаются здесь. Внешние YAML/JSON-конфиги не используются.

BENCHMARK_PARAMS = {
    "output_dir": "benchmark_data",
    "sigma": 1.0,
    "epsilon": 1.0,
    "mass": 1.0,
    "rcut": 2.5,
    "dt": 0.002,
    "friction": 1.0,
    "precision": "mixed",
    "force_platform": "auto",
    "default_device_index": 0,
    "speed_N_values": [1024, 2048, 4096],
    "speed_T": 1.0,
    "speed_rho": 0.08,
    "speed_steps": 10_000,
    "speed_warmup_steps": 1_000,
    "speed_seed": 20260619,
    "multigpu_N": 1024,
    "multigpu_T": 1.0,
    "multigpu_rho": 0.08,
    "multigpu_steps": 10_000,
    "multigpu_warmup_steps": 1_000,
    "pilot_N": 1024,
    "pilot_steps": 200_000,
    "pilot_sample_interval": 1_000,
    "pilot_block_steps": 10_000,
    "pilot_seed": 20260619,
    "representative_points": [
        {"T": 1.10, "rho": 0.02, "label": "hot_gas"},
        {"T": 1.10, "rho": 0.16, "label": "hot_moderate_dense"},
        {"T": 1.00, "rho": 0.12, "label": "fit_work_area"},
        {"T": 0.90, "rho": 0.12, "label": "slow_transition"},
        {"T": 0.80, "rho": 0.16, "label": "lowT_dense"},
        {"T": 0.70, "rho": 0.08, "label": "plateau_area"},
        {"T": 0.70, "rho": 0.16, "label": "hard_lowT_dense"},
    ],
    "tail_blocks_for_reference": 3,
    "drift_relative_tolerance": 0.05,
    "drift_sigma_tolerance": 1.5,
    "target_pressure_sem": 0.03,
    "min_prod_blocks": 4,
    "runtime_scenarios": [
        {"scenario_name": "pilot_grid", "N": 1024, "n_temperatures": 5, "n_densities": 8, "n_seeds": 2},
        {"scenario_name": "production_medium", "N": 2048, "n_temperatures": 7, "n_densities": 12, "n_seeds": 3},
        {"scenario_name": "production_large", "N": 4096, "n_temperatures": 9, "n_densities": 16, "n_seeds": 4},
    ],
}

if os.environ.get("LJ_BENCH_SMOKE") == "1":
    BENCHMARK_PARAMS.update({
        "speed_N_values": [32],
        "speed_steps": 20,
        "speed_warmup_steps": 2,
        "multigpu_N": 32,
        "multigpu_steps": 20,
        "multigpu_warmup_steps": 2,
        "pilot_N": 32,
        "pilot_steps": 40,
        "pilot_sample_interval": 10,
        "pilot_block_steps": 20,
        "representative_points": [{"T": 1.00, "rho": 0.08, "label": "smoke"}],
        "runtime_scenarios": [{"scenario_name": "smoke", "N": 32, "n_temperatures": 1, "n_densities": 1, "n_seeds": 1}],
    })
    print("LJ_BENCH_SMOKE=1: using tiny benchmark parameters.")

OUTPUT_DIR = Path(BENCHMARK_PARAMS["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BENCHMARK_PARAMS


## Секция 1. Проверка окружения и CUDA

Эта секция уже выполнена первой ячейкой: показан nvidia-smi, список OpenMM platforms, наличие CUDA и краткая информация по GPU. Следующая ячейка фиксирует выбор платформы и проверяет DeviceIndex.


In [ ]:
def available_platforms():
    return [mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]

def select_openmm_platform(device_index=None, precision=None, force_platform="auto"):
    names = available_platforms()
    requested = force_platform
    if requested == "auto":
        if "CUDA" in names:
            requested = "CUDA"
        elif "CPU" in names:
            requested = "CPU"
        else:
            requested = names[0]
    if requested not in names:
        raise RuntimeError(f"Requested OpenMM platform {requested!r} is unavailable. Available: {names}")

    platform = mm.Platform.getPlatformByName(requested)
    properties = {}
    if requested == "CUDA":
        properties["Precision"] = str(precision or BENCHMARK_PARAMS["precision"])
        if device_index is not None:
            properties["DeviceIndex"] = str(device_index)
    return platform, properties

def gpu_name(device_index):
    if device_index is None:
        return ""
    return GPU_NAMES.get(int(device_index), f"GPU {device_index}")

def print_platform_check(device_index=None):
    platform, properties = select_openmm_platform(
        device_index=device_index,
        precision=BENCHMARK_PARAMS["precision"],
        force_platform=BENCHMARK_PARAMS["force_platform"],
    )
    print("Selected OpenMM platform:", platform.getName())
    print("Selected properties:", properties)
    if platform.getName() != "CUDA":
        print("WARNING: selected platform is not CUDA; timing is not a DataSphere GPU estimate.")
    elif device_index is not None:
        print(f"CUDA DeviceIndex={device_index}; gpu_name={gpu_name(device_index)}")
    return platform, properties

DEFAULT_DEVICE_INDEX = BENCHMARK_PARAMS["default_device_index"]
if AVAILABLE_GPU_INDICES and DEFAULT_DEVICE_INDEX not in AVAILABLE_GPU_INDICES:
    DEFAULT_DEVICE_INDEX = AVAILABLE_GPU_INDICES[0]
    print("Default DeviceIndex adjusted to available GPU:", DEFAULT_DEVICE_INDEX)

DEFAULT_PLATFORM, DEFAULT_PROPERTIES = print_platform_check(DEFAULT_DEVICE_INDEX if CUDA_AVAILABLE else None)


## Минимальная LJ/OpenMM-логика

Только функции, нужные для бенчмарков: создание LJ-системы, начальная расстановка частиц, запуск точки на выбранном GPU, sampling step,T,P,U,K,E, блоки, drift/SEM и запись CSV.


In [ ]:
SPEED_COLUMNS = [
    "benchmark_id", "N", "T", "rho", "steps", "dt", "device_index", "platform",
    "precision", "gpu_name", "wall_time_s", "steps_per_second", "notes",
]
MULTIGPU_COLUMNS = [
    "benchmark_id", "n_gpus_requested", "device_index", "N", "T", "rho", "steps",
    "wall_time_s", "steps_per_second", "success", "error_message",
]
TRACE_COLUMNS = [
    "bench_id", "T_target", "rho_target", "seed", "step", "T_inst", "P_inst",
    "U_inst", "K_inst", "E_inst",
]
BLOCK_COLUMNS = [
    "bench_id", "T_target", "rho_target", "seed", "block", "step_start", "step_end",
    "n_samples", "T_mean", "P_mean", "U_mean", "K_mean", "E_mean", "T_std", "P_std", "U_std",
]
SUMMARY_COLUMNS = [
    "bench_id", "T_target", "rho_target", "seed", "suggested_equil_steps",
    "suggested_prod_steps", "P_sem_est", "U_drift_flag", "P_drift_flag", "notes",
]
RUNTIME_COLUMNS = [
    "scenario_name", "N", "n_temperatures", "n_densities", "n_seeds", "n_points",
    "equil_steps", "prod_steps", "estimated_time_1gpu_h", "estimated_time_ngpu_h",
    "n_gpus", "assumptions",
]

def save_csv(rows, columns, filename):
    path = OUTPUT_DIR / filename
    df = pd.DataFrame(rows, columns=columns)
    df.to_csv(path, index=False)
    print(f"Saved {path} ({len(df)} rows)")
    return df

def reduced_temperature_to_openmm(T_reduced, epsilon):
    gas_r = unit.MOLAR_GAS_CONSTANT_R.value_in_unit(unit.kilojoule_per_mole / unit.kelvin)
    return (float(T_reduced) * float(epsilon) / gas_r) * unit.kelvin

def initialize_positions(N, L, seed=None, jitter_fraction=0.03):
    n_side = math.ceil(N ** (1.0 / 3.0))
    spacing = L / n_side
    coords = []
    for ix in range(n_side):
        for iy in range(n_side):
            for iz in range(n_side):
                if len(coords) == N:
                    coords = np.asarray(coords, dtype=float)
                    if seed is not None and jitter_fraction > 0:
                        rng = np.random.default_rng(seed)
                        coords += rng.uniform(-0.5, 0.5, size=coords.shape) * spacing * jitter_fraction
                        coords %= L
                    return coords
                coords.append([(ix + 0.5) * spacing, (iy + 0.5) * spacing, (iz + 0.5) * spacing])
    return np.asarray(coords, dtype=float)

def make_topology(N):
    topology = app.Topology()
    chain = topology.addChain()
    residue = topology.addResidue("LJ", chain)
    element = app.Element.getByAtomicNumber(18)
    for _ in range(N):
        topology.addAtom("Ar", element, residue)
    return topology

def create_lj_system(N, rho, params):
    sigma = float(params["sigma"])
    epsilon = float(params["epsilon"])
    mass = float(params["mass"])
    rcut = float(params["rcut"])
    L = float((int(N) / float(rho)) ** (1.0 / 3.0))

    system = mm.System()
    for _ in range(int(N)):
        system.addParticle(mass * unit.dalton)

    system.setDefaultPeriodicBoxVectors(
        unit.Quantity(mm.Vec3(L, 0, 0), unit.nanometer),
        unit.Quantity(mm.Vec3(0, L, 0), unit.nanometer),
        unit.Quantity(mm.Vec3(0, 0, L), unit.nanometer),
    )

    expr = "4*epsilon*((sigma/r)^12-(sigma/r)^6)-4*epsilon*((sigma/rcut)^12-(sigma/rcut)^6)"
    force = mm.CustomNonbondedForce(expr)
    force.addGlobalParameter("sigma", sigma * unit.nanometer)
    force.addGlobalParameter("epsilon", epsilon * unit.kilojoule_per_mole)
    force.addGlobalParameter("rcut", rcut * unit.nanometer)
    force.setNonbondedMethod(mm.CustomNonbondedForce.CutoffPeriodic)
    force.setCutoffDistance(rcut * unit.nanometer)
    force.setUseLongRangeCorrection(False)
    for _ in range(int(N)):
        force.addParticle([])
    system.addForce(force)

    system.addForce(mm.CMMotionRemover(100))
    return system, L

def build_simulation(N, T, rho, seed, device_index=None, params=BENCHMARK_PARAMS):
    system, L = create_lj_system(N, rho, params)
    openmm_T = reduced_temperature_to_openmm(T, params["epsilon"])
    integrator = mm.LangevinMiddleIntegrator(
        openmm_T,
        float(params["friction"]) / unit.picosecond,
        float(params["dt"]) * unit.picoseconds,
    )
    integrator.setRandomNumberSeed(int(seed))

    platform, properties = select_openmm_platform(
        device_index=device_index,
        precision=params["precision"],
        force_platform=params["force_platform"],
    )
    simulation = app.Simulation(make_topology(N), system, integrator, platform, properties)
    simulation.context.setPositions(initialize_positions(N, L, seed=seed) * unit.nanometer)
    simulation.context.setVelocitiesToTemperature(openmm_T, int(seed))
    actual_platform = simulation.context.getPlatform().getName()
    print(f"OpenMM simulation platform: {actual_platform}; DeviceIndex={device_index}; N={N}; T={T}; rho={rho}")
    return simulation, L, actual_platform

def get_positions_nm(simulation):
    state = simulation.context.getState(getPositions=True)
    return np.asarray(state.getPositions(asNumpy=True).value_in_unit(unit.nanometer), dtype=float)

def lj_virial_pressure(positions, L, T_inst, params=BENCHMARK_PARAMS):
    sigma = float(params["sigma"])
    epsilon = float(params["epsilon"])
    rcut = float(params["rcut"])
    V = L ** 3
    rho = len(positions) / V
    virial = 0.0
    rcut2 = rcut * rcut

    for i in range(len(positions) - 1):
        delta = positions[i + 1:] - positions[i]
        delta -= L * np.rint(delta / L)
        r2 = np.sum(delta * delta, axis=1)
        mask = (r2 > 0.0) & (r2 < rcut2)
        if not np.any(mask):
            continue
        inv_r2 = (sigma * sigma) / r2[mask]
        inv_r6 = inv_r2 ** 3
        inv_r12 = inv_r6 ** 2
        virial += float(np.sum(24.0 * epsilon * (2.0 * inv_r12 - inv_r6)))
    return float(rho * T_inst + virial / (3.0 * V))

def sample_observables(simulation, N, L, step, T_target, rho_target, seed, bench_id):
    state = simulation.context.getState(getEnergy=True)
    K = state.getKineticEnergy().value_in_unit(unit.kilojoule_per_mole)
    U = state.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
    T_inst = 2.0 * K / (max(1, 3 * int(N) - 3) * float(BENCHMARK_PARAMS["epsilon"]))
    positions = get_positions_nm(simulation)
    P_inst = lj_virial_pressure(positions, L, T_inst)
    return {
        "bench_id": bench_id,
        "T_target": float(T_target),
        "rho_target": float(rho_target),
        "seed": int(seed),
        "step": int(step),
        "T_inst": float(T_inst),
        "P_inst": float(P_inst),
        "U_inst": float(U),
        "K_inst": float(K),
        "E_inst": float(U + K),
    }

def mean_or_nan(values):
    vals = [float(v) for v in values if math.isfinite(float(v))]
    return float(mean(vals)) if vals else math.nan

def std_or_zero(values):
    vals = [float(v) for v in values if math.isfinite(float(v))]
    if len(vals) < 2:
        return 0.0
    return float(stdev(vals))

def sem_or_zero(values):
    vals = [float(v) for v in values if math.isfinite(float(v))]
    if len(vals) < 2:
        return 0.0
    return float(stdev(vals) / math.sqrt(len(vals)))


## Секция 2. Speed benchmark

Короткие прогоны для разных N, одна строка на прогон. Основной результат: benchmark_data/speed_benchmark.csv.


In [ ]:
def run_speed_case(benchmark_id, N, T, rho, steps, warmup_steps, seed, device_index=None):
    notes = ""
    if not CUDA_AVAILABLE:
        notes = "CPU result; not suitable for DataSphere GPU runtime estimate"
    simulation, _L, actual_platform = build_simulation(N, T, rho, seed, device_index=device_index)
    if warmup_steps:
        simulation.step(int(warmup_steps))

    start = time.perf_counter()
    simulation.step(int(steps))
    wall_time_s = time.perf_counter() - start
    steps_per_second = float(steps) / wall_time_s if wall_time_s > 0 else math.nan

    return {
        "benchmark_id": benchmark_id,
        "N": int(N),
        "T": float(T),
        "rho": float(rho),
        "steps": int(steps),
        "dt": float(BENCHMARK_PARAMS["dt"]),
        "device_index": "" if device_index is None else int(device_index),
        "platform": actual_platform,
        "precision": BENCHMARK_PARAMS["precision"] if actual_platform == "CUDA" else "",
        "gpu_name": gpu_name(device_index) if actual_platform == "CUDA" else "",
        "wall_time_s": float(wall_time_s),
        "steps_per_second": float(steps_per_second),
        "notes": notes,
    }

speed_rows = []
speed_device = DEFAULT_DEVICE_INDEX if CUDA_AVAILABLE else None
for N in tqdm(BENCHMARK_PARAMS["speed_N_values"], desc="speed benchmark"):
    benchmark_id = f"speed_N{int(N)}_dev{speed_device if speed_device is not None else 'cpu'}"
    row = run_speed_case(
        benchmark_id=benchmark_id,
        N=int(N),
        T=BENCHMARK_PARAMS["speed_T"],
        rho=BENCHMARK_PARAMS["speed_rho"],
        steps=BENCHMARK_PARAMS["speed_steps"],
        warmup_steps=BENCHMARK_PARAMS["speed_warmup_steps"],
        seed=BENCHMARK_PARAMS["speed_seed"] + int(N),
        device_index=speed_device,
    )
    speed_rows.append(row)

speed_df = save_csv(speed_rows, SPEED_COLUMNS, "speed_benchmark.csv")
speed_df


## Секция 3. Multi-GPU benchmark

Стратегия для будущего DataSphere-run: одна независимая EOS-точка на одну GPU. Эта секция не пытается заставить одну OpenMM simulation использовать все GPU.


In [ ]:
def _multigpu_worker(device_index, params):
    try:
        benchmark_id = f"multigpu_dev{device_index}"
        row = run_speed_case(
            benchmark_id=benchmark_id,
            N=int(params["multigpu_N"]),
            T=float(params["multigpu_T"]),
            rho=float(params["multigpu_rho"]),
            steps=int(params["multigpu_steps"]),
            warmup_steps=int(params["multigpu_warmup_steps"]),
            seed=int(params["speed_seed"]) + int(device_index),
            device_index=int(device_index),
        )
        return {
            "benchmark_id": benchmark_id,
            "n_gpus_requested": len(params["_gpu_indices"]),
            "device_index": int(device_index),
            "N": int(params["multigpu_N"]),
            "T": float(params["multigpu_T"]),
            "rho": float(params["multigpu_rho"]),
            "steps": int(params["multigpu_steps"]),
            "wall_time_s": row["wall_time_s"],
            "steps_per_second": row["steps_per_second"],
            "success": True,
            "error_message": "",
        }
    except Exception as exc:
        return {
            "benchmark_id": f"multigpu_dev{device_index}",
            "n_gpus_requested": len(params.get("_gpu_indices", [])),
            "device_index": int(device_index),
            "N": int(params["multigpu_N"]),
            "T": float(params["multigpu_T"]),
            "rho": float(params["multigpu_rho"]),
            "steps": int(params["multigpu_steps"]),
            "wall_time_s": math.nan,
            "steps_per_second": math.nan,
            "success": False,
            "error_message": repr(exc),
        }

def run_multigpu_benchmark():
    if not CUDA_AVAILABLE:
        print("Multi-GPU benchmark skipped: CUDA is not visible to OpenMM.")
        return save_csv([], MULTIGPU_COLUMNS, "multigpu_benchmark.csv")

    gpu_indices = AVAILABLE_GPU_INDICES
    if len(gpu_indices) <= 1:
        print("Multi-GPU benchmark skipped: only one GPU is available.")
        return save_csv([], MULTIGPU_COLUMNS, "multigpu_benchmark.csv")

    params = dict(BENCHMARK_PARAMS)
    params["_gpu_indices"] = gpu_indices

    rows = []
    try:
        ctx = mp.get_context("fork")
        with ProcessPoolExecutor(max_workers=len(gpu_indices), mp_context=ctx) as pool:
            futures = [pool.submit(_multigpu_worker, idx, params) for idx in gpu_indices]
            for future in as_completed(futures):
                rows.append(future.result())
    except Exception as exc:
        print("Parallel multi-GPU launch failed; falling back to sequential workers:", repr(exc))
        rows = [_multigpu_worker(idx, params) for idx in gpu_indices]

    df = save_csv(sorted(rows, key=lambda r: r["device_index"]), MULTIGPU_COLUMNS, "multigpu_benchmark.csv")
    single_gpu_best = speed_df["steps_per_second"].max() if len(speed_df) else math.nan
    total_multigpu = df.loc[df["success"], "steps_per_second"].sum() if len(df) else 0.0
    print(f"Best single-GPU speed: {single_gpu_best:.3g} steps/s")
    print(f"Sum of independent worker speeds: {total_multigpu:.3g} steps/s")
    return df

multigpu_df = run_multigpu_benchmark()
multigpu_df


## Секция 4. Equilibration и production benchmark

Pilot-прогоны для representative points. Сохраняются только редкие samples и блочные средние, без траекторий/координат/профилей.


In [ ]:
def block_trace_rows(trace_rows, block_steps):
    if not trace_rows:
        return []

    df = pd.DataFrame(trace_rows)
    first_step = int(df["step"].min())
    df["block"] = ((df["step"] - first_step) // int(block_steps)).astype(int)

    block_rows = []
    for block, part in df.groupby("block", sort=True):
        block_rows.append({
            "bench_id": part["bench_id"].iloc[0],
            "T_target": float(part["T_target"].iloc[0]),
            "rho_target": float(part["rho_target"].iloc[0]),
            "seed": int(part["seed"].iloc[0]),
            "block": int(block),
            "step_start": int(part["step"].min()),
            "step_end": int(part["step"].max()),
            "n_samples": int(len(part)),
            "T_mean": mean_or_nan(part["T_inst"]),
            "P_mean": mean_or_nan(part["P_inst"]),
            "U_mean": mean_or_nan(part["U_inst"]),
            "K_mean": mean_or_nan(part["K_inst"]),
            "E_mean": mean_or_nan(part["E_inst"]),
            "T_std": std_or_zero(part["T_inst"]),
            "P_std": std_or_zero(part["P_inst"]),
            "U_std": std_or_zero(part["U_inst"]),
        })
    return block_rows

def has_tail_drift(values, relative_tolerance, sigma_tolerance):
    vals = np.asarray([float(v) for v in values if math.isfinite(float(v))], dtype=float)
    if len(vals) < 4:
        return False
    x = np.arange(len(vals), dtype=float)
    slope = np.polyfit(x, vals, 1)[0]
    span_change = abs(float(slope) * max(1, len(vals) - 1))
    scale = max(abs(float(np.mean(vals))) * relative_tolerance, float(np.std(vals)) * sigma_tolerance, 1e-12)
    return bool(span_change > scale)

def suggest_equil_and_prod(block_rows, params=BENCHMARK_PARAMS):
    if not block_rows:
        return {
            "suggested_equil_steps": int(params["pilot_steps"] // 2),
            "suggested_prod_steps": int(params["pilot_block_steps"] * params["min_prod_blocks"]),
            "P_sem_est": math.nan,
            "U_drift_flag": True,
            "P_drift_flag": True,
            "notes": "no block rows",
        }

    blocks = pd.DataFrame(block_rows).sort_values("block").reset_index(drop=True)
    tail_n = min(int(params["tail_blocks_for_reference"]), len(blocks))
    tail = blocks.tail(tail_n)

    u_tail_mean = mean_or_nan(tail["U_mean"])
    p_tail_mean = mean_or_nan(tail["P_mean"])
    u_tail_std = std_or_zero(tail["U_mean"])
    p_tail_std = std_or_zero(tail["P_mean"])

    rel_tol = float(params["drift_relative_tolerance"])
    sig_tol = float(params["drift_sigma_tolerance"])

    suggested_equil = int(blocks.iloc[min(len(blocks) - 1, max(0, len(blocks) // 2))]["step_start"])
    for idx in range(len(blocks)):
        window = blocks.iloc[idx:]
        u_window = mean_or_nan(window["U_mean"])
        p_window = mean_or_nan(window["P_mean"])
        u_scale = max(abs(u_tail_mean) * rel_tol, u_tail_std * sig_tol, 1e-12)
        p_scale = max(abs(p_tail_mean) * rel_tol, p_tail_std * sig_tol, 1e-12)
        if abs(u_window - u_tail_mean) <= u_scale and abs(p_window - p_tail_mean) <= p_scale:
            suggested_equil = int(blocks.iloc[idx]["step_start"])
            break

    prod_blocks = blocks[blocks["step_start"] >= suggested_equil].copy()
    if len(prod_blocks) < int(params["min_prod_blocks"]):
        prod_blocks = blocks.tail(int(params["min_prod_blocks"])).copy()

    p_sem = sem_or_zero(prod_blocks["P_mean"])
    target_sem = float(params["target_pressure_sem"])
    block_steps = int(params["pilot_block_steps"])
    suggested_prod = int(max(len(prod_blocks), int(params["min_prod_blocks"])) * block_steps)

    if math.isfinite(p_sem) and p_sem > target_sem and target_sem > 0 and len(prod_blocks) >= 2:
        scale = min(25.0, (p_sem / target_sem) ** 2)
        suggested_prod = int(math.ceil(suggested_prod * scale / block_steps) * block_steps)

    tail_for_drift = blocks.tail(max(4, tail_n))
    u_drift_flag = has_tail_drift(tail_for_drift["U_mean"], rel_tol, sig_tol)
    p_drift_flag = has_tail_drift(tail_for_drift["P_mean"], rel_tol, sig_tol)

    notes = []
    if u_drift_flag:
        notes.append("U tail drift")
    if p_drift_flag:
        notes.append("P tail drift")
    if not notes:
        notes.append("ok")

    return {
        "suggested_equil_steps": int(suggested_equil),
        "suggested_prod_steps": int(suggested_prod),
        "P_sem_est": float(p_sem),
        "U_drift_flag": bool(u_drift_flag),
        "P_drift_flag": bool(p_drift_flag),
        "notes": "; ".join(notes),
    }

def run_pilot_point(point, seed, device_index=None):
    T = float(point["T"])
    rho = float(point["rho"])
    label = point.get("label", f"T{T}_rho{rho}")
    bench_id = f"pilot_{label}_T{T:.2f}_rho{rho:.3f}_seed{seed}".replace(".", "p")

    N = int(BENCHMARK_PARAMS["pilot_N"])
    pilot_steps = int(BENCHMARK_PARAMS["pilot_steps"])
    sample_interval = int(BENCHMARK_PARAMS["pilot_sample_interval"])

    simulation, L, _actual_platform = build_simulation(N, T, rho, seed, device_index=device_index)

    trace_rows = []
    completed = 0
    while completed < pilot_steps:
        chunk = min(sample_interval, pilot_steps - completed)
        simulation.step(int(chunk))
        completed += int(chunk)
        row = sample_observables(simulation, N, L, completed, T, rho, seed, bench_id)
        trace_rows.append(row)
        if not math.isfinite(row["T_inst"]) or row["T_inst"] > 10.0 * T:
            print(f"Stopping {bench_id}: unstable T_inst={row['T_inst']}")
            break

    block_rows = block_trace_rows(trace_rows, BENCHMARK_PARAMS["pilot_block_steps"])
    rec = suggest_equil_and_prod(block_rows)
    summary_row = {
        "bench_id": bench_id,
        "T_target": T,
        "rho_target": rho,
        "seed": int(seed),
        **rec,
    }
    return trace_rows, block_rows, summary_row

pilot_device = DEFAULT_DEVICE_INDEX if CUDA_AVAILABLE else None
all_trace_rows = []
all_block_rows = []
summary_rows = []

for point in tqdm(BENCHMARK_PARAMS["representative_points"], desc="pilot benchmark"):
    trace_rows, block_rows, summary_row = run_pilot_point(
        point,
        seed=BENCHMARK_PARAMS["pilot_seed"],
        device_index=pilot_device,
    )
    all_trace_rows.extend(trace_rows)
    all_block_rows.extend(block_rows)
    summary_rows.append(summary_row)

trace_df = save_csv(all_trace_rows, TRACE_COLUMNS, "equilibration_traces.csv")
blocks_df = save_csv(all_block_rows, BLOCK_COLUMNS, "equilibration_blocks.csv")
summary_df = save_csv(summary_rows, SUMMARY_COLUMNS, "benchmark_summary.csv")

summary_df


## Runtime estimates и итоговый summary

Оценки времени используют измеренную скорость ближайшего N из speed benchmark и рекомендации из pilot-прогонов.


In [ ]:
def nearest_speed_for_N(N, speed_df):
    if len(speed_df) == 0:
        return math.nan
    tmp = speed_df.copy()
    tmp["N_distance"] = (tmp["N"].astype(float) - float(N)).abs()
    row = tmp.sort_values(["N_distance", "steps_per_second"], ascending=[True, False]).iloc[0]
    return float(row["steps_per_second"])

def conservative_recommendations(summary_df):
    if len(summary_df) == 0:
        return int(BENCHMARK_PARAMS["pilot_steps"] // 2), int(BENCHMARK_PARAMS["pilot_steps"])
    equil = int(math.ceil(float(summary_df["suggested_equil_steps"].max()) / 1000.0) * 1000)
    prod = int(math.ceil(float(summary_df["suggested_prod_steps"].max()) / 1000.0) * 1000)
    return equil, prod

recommended_equil, recommended_prod = conservative_recommendations(summary_df)

runtime_rows = []
n_gpus_for_estimate = max(1, len(AVAILABLE_GPU_INDICES)) if CUDA_AVAILABLE else 1
for scenario in BENCHMARK_PARAMS["runtime_scenarios"]:
    N = int(scenario["N"])
    n_points = int(scenario["n_temperatures"]) * int(scenario["n_densities"]) * int(scenario["n_seeds"])
    sps = nearest_speed_for_N(N, speed_df)
    total_steps_per_point = recommended_equil + recommended_prod
    estimated_1gpu_h = (n_points * total_steps_per_point / sps / 3600.0) if sps and math.isfinite(sps) and sps > 0 else math.nan
    estimated_ngpu_h = estimated_1gpu_h / n_gpus_for_estimate if math.isfinite(estimated_1gpu_h) else math.nan
    runtime_rows.append({
        "scenario_name": scenario["scenario_name"],
        "N": N,
        "n_temperatures": int(scenario["n_temperatures"]),
        "n_densities": int(scenario["n_densities"]),
        "n_seeds": int(scenario["n_seeds"]),
        "n_points": int(n_points),
        "equil_steps": int(recommended_equil),
        "prod_steps": int(recommended_prod),
        "estimated_time_1gpu_h": float(estimated_1gpu_h),
        "estimated_time_ngpu_h": float(estimated_ngpu_h),
        "n_gpus": int(n_gpus_for_estimate),
        "assumptions": (
            f"nearest measured speed for N={N}; total_steps_per_point=equil+prod; "
            "independent EOS points distributed one per GPU; no trajectory/profile output"
        ),
    })

runtime_df = save_csv(runtime_rows, RUNTIME_COLUMNS, "runtime_estimates.csv")
runtime_df


In [ ]:
print("\n===== BENCHMARK SUMMARY =====")
if len(speed_df):
    best = speed_df.sort_values("steps_per_second", ascending=False).iloc[0]
    print(f"Best measured single-worker speed: {best.steps_per_second:.3g} steps/s "
          f"(N={int(best.N)}, platform={best.platform}, gpu={best.gpu_name or 'n/a'})")
else:
    print("No speed benchmark rows.")

if len(multigpu_df):
    ok = multigpu_df[multigpu_df["success"]]
    print(f"Multi-GPU workers: {len(ok)}/{len(multigpu_df)} successful; "
          f"sum speed={ok['steps_per_second'].sum():.3g} steps/s")
else:
    print("Multi-GPU benchmark: skipped or no rows.")

if len(speed_df):
    recommended_N = int(speed_df.sort_values("steps_per_second", ascending=False).iloc[0]["N"])
    print(f"Recommended N for first production planning pass: {recommended_N}")
else:
    recommended_N = BENCHMARK_PARAMS["pilot_N"]
    print(f"Recommended N fallback: {recommended_N}")

print(f"Recommended equil_steps: {recommended_equil}")
print(f"Recommended prod_steps: {recommended_prod}")

print("\nRuntime estimates:")
print(runtime_df.to_string(index=False))

if not CUDA_AVAILABLE:
    print("\nWARNING: CUDA was not visible. These CPU timings are useful only for notebook correctness, "
          "not for DataSphere GPU runtime planning.")

print("\nFuture acquisition notebook criteria:")
print("1. EOS points only: no fit, plots, profiles, or visualization.")
print("2. Use CUDA/OpenMM and explicit DeviceIndex.")
print("3. Parallelize independent EOS points across GPUs; do not put one simulation on all GPUs.")
print("4. Save final rows and block means, not full time series.")
print("5. Resume by run_id: skip status=ok.")
print("6. Write shard files per GPU/worker.")
print("7. Save failures.csv and continue after individual point failures.")
print("8. Choose N/equil/prod/sample_interval/n_blocks/grid from these benchmark CSVs.")
